In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import kendalltau
from itertools import combinations
import re
import matplotlib.pyplot as plt
from mne.viz import circular_layout
from mne_connectivity.viz import plot_connectivity_circle
from statsmodels.stats.multitest import multipletests

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Calibri'] + plt.rcParams['font.sans-serif']

LONG_CSV = '/path/to/avg_ig_by_region copy.csv'
VALUE_COL = 'avg_saliency'

long_df = pd.read_csv(LONG_CSV)
lookup = pd.read_csv('/path/to/fs_lookup.csv', usecols=['code', 'region'])
lobes = pd.read_csv('/path/to/destrieux - lobe.csv')

area = list(lobes['area'])
lharea = ['lh_' + i for i in area]
rharea = ['rh_' + i for i in area]
rharea = rharea[::-1]
area = lharea + rharea

labels_ref = area

data = long_df.pivot_table(index='subject', columns='region_id',
                           values=VALUE_COL, aggfunc='first', dropna=False)

code_to_region = dict(zip(lookup['code'], lookup['region']))
data.columns = [re.sub(r'^ctx_', '', code_to_region.get(c, str(c))) for c in data.columns]
data = data.apply(pd.to_numeric, errors='coerce')

EXCLUDE = {'medial_wall', 'unknown'}   

def is_excluded(region_name):
    return re.sub(r'^(lh|rh)_', '', region_name).lower() in EXCLUDE

available = [l for l in labels_ref if l in data.columns and not is_excluded(l)]
missing   = [l for l in labels_ref if l not in data.columns]

dropped = [l for l in labels_ref if l in data.columns and is_excluded(l)]
if dropped:
    print(f"Excluded {len(dropped)} regions:", dropped)
if missing:
    print(f"{len(missing)} regions in labels_ref not found in data — dropped:", missing)

data = data[available]
labels = available

area_to_lobe = dict(zip(lobes['area'], lobes['lobe']))

def strip_hemisphere(region_name):
    return re.sub(r'^(lh|rh)_', '', region_name)

def get_rgba_value(number):
    if str(number) == "frontal":
        return (234/255,67/255,53/255, 255/255)
    elif str(number) == "parietal":
        return (227/255,116/255,0/255, 255/255)
    elif str(number) == "occipital":
        return (66/255,103/255,210/255, 255/255)
    elif str(number) == "temporal":
        return (52/255,168/255,83/255, 255/255)
    elif str(number) == "limbic":
        return (255/255,194/255,0/255, 255/255)
    elif int(number) == 5:
        return (251/255,188/255,4/255,255/255)

new_lobe_order = ['frontal', 'temporal', 'parietal', 'occipital', 'limbic']

def region_lobe(region_name):
    return area_to_lobe.get(strip_hemisphere(region_name))

lh_labels = [l for l in labels if l.startswith('lh_')]
rh_labels = [l for l in labels if l.startswith('rh_')]

lh_sorted = []
for lobe in new_lobe_order:
    lh_sorted += [l for l in lh_labels if region_lobe(l) == lobe]

rh_sorted = []
for lobe in reversed(new_lobe_order):
    rh_sorted += [l for l in rh_labels if region_lobe(l) == lobe]

labels = lh_sorted + rh_sorted
colors = [get_rgba_value(region_lobe(l)) for l in labels]


def build_connectivity(data, labels, alpha=0.05, method='fdr_bh',
                       positive_only=True, min_n=3):
    n = len(labels)
    pairs, taus, pvals = [], [], []

    for i, j in combinations(range(n), 2):
        xi, yj = labels[i], labels[j]
        pair = data[[xi, yj]].dropna()
        if len(pair) < min_n:
            continue
        tau, p = kendalltau(pair[xi], pair[yj])
        if np.isnan(p):
            continue
        pairs.append((i, j))
        taus.append(tau)
        pvals.append(p)

    taus = np.asarray(taus)
    pvals = np.asarray(pvals)

    reject, qvals, _, _ = multipletests(pvals, alpha=alpha, method=method)

    keep = reject & (taus > 0) if positive_only else reject

    conn = np.zeros((n, n))
    for (i, j), t, k in zip(pairs, taus, keep):
        if k:
            conn[i, j] = conn[j, i] = abs(t)

    print(f"{keep.sum()} / {len(pvals)} pairs survive {method} at q<{alpha}")
    return conn, pairs, taus, pvals, qvals

data = data[labels]
n = len(labels)
connectivity_array, pairs, taus, pvals, qvals = build_connectivity(data, labels)


fig, ax = plt.subplots(figsize=(12, 12), facecolor="white", subplot_kw=dict(polar=True), dpi=800)
plot_connectivity_circle(con=connectivity_array,
                                     node_names=labels,
                                     n_lines=1000,
                                     node_angles=circular_layout(labels,labels,90),
                                     node_colors=colors,
                                     facecolor='white',
                                     textcolor='black',
                                     colormap='YlGnBu',
                                     colorbar_pos=(0.75, -0.15),
                                     vmax=1.0,
                                     vmin=0.5,
                                     fontsize_names=10,
                                     padding=1,
                                     node_linewidth=1,
                                     ax=ax
)